# Triage de Datos y Pipelines en Machine Learning Clínico

En los hospitales, cuando llega una ambulancia, el paciente crítico pasa por el **Triage** (Admisión, Medición de Signos, Estabilización) antes de entrar a Cirugía.

En Ciencia de Datos Biomédica ocurre exactamente igual con nuestros Pipeline de Machine Learning para predecir cáncer o ataques cardíacos. Aquí veremos cómo preparar estos datos quirúrgicamente usando: **ETL**, **EDA**, **Preprocesamiento** e **Sklearn Pipelines**.

---

## 1. ETL Médico (Extract, Transform, Load)

Los Monitores Multiparamétricos (ECG, Oxímetro) a menudo registran fallos (Errores de hardware o pérdida de señal). Nuestra tarea es extraer estos más de mil registros, transformarlos y dejarlos usables.

In [ ]:
import pandas as pd
import numpy as np

# 1.1 EXTRACCIÓN: Datos crudos desde los sistemas del Hospital (con errores típicos)
datos_hospital = {
    'Paciente_ID': [101, 102, 103, 104, 105],
    'Edad': [55, 60, np.nan, 32, 70], # np.nan = A enfermería se le olvidó llenarlo
    'PresionArterial': ['120mmHg', '140mmHg', 'ERROR_SENSOR', '115mmHg', '160mmHg'],
    'Glucosa_Sangre': [100.5, 95.0, 110.2, 88.5, 130.0],
    'Enfermedad_Cardiaca': ['Si', 'Si', 'No', 'No', 'Si']
}
df_extraido = pd.DataFrame(datos_hospital)
print("---- Datos Recién Obtenidos de la UCI ----")
display(df_extraido)

# 1.2 TRANSFORMACIÓN: Quitar el texto 'mmHg' para volverlo un número puro, omitiendo errores.
df_transformado = df_extraido.copy()
df_transformado['Presion_Sistolica'] = df_transformado['PresionArterial'].str.replace('mmHg', '', regex=False)
df_transformado['Presion_Sistolica'] = pd.to_numeric(df_transformado['Presion_Sistolica'], errors='coerce')
df_transformado = df_transformado.drop(columns=['PresionArterial'])

print("\n---- Datos Transformados en Cifras Médicas ----")
display(df_transformado)

# 1.3 CARGA (Load): Salvar a una base de datos validada de pacientes.
df_transformado.to_csv('signos_vitales_limpios.csv', index=False)

---

## 2. EDA: Análisis Exploratorio de Diagnósticos

Antes de predecir la Enfermedad Cardíaca de cientos de personas, debemos generar estadísticas descriptivas y visuales (gráficos epidemológicos) para dar con anomalías clínicas obvias en las analíticas de sangre.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Generando mil pacientes de muestra para simulación médica
np.random.seed(42)
datos_eda = pd.DataFrame({
    'Edad': np.random.normal(55, 12, 1000),
    'Colesterol': np.random.normal(210, 40, 1000),
    'Estado_Cardiovascular': np.random.choice(['Sano', 'En_Riesgo', 'Hipertenso'], 1000)
})

print("Estadisticas Poblacionales (Avisos de taquicardias y Lípidos Altos):")
display(datos_eda.describe())

# Histograma de Edades en la cohorte clínica
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.histplot(datos_eda['Edad'], kde=True, color='brown', bins=20)
plt.title('Demografía de Edades en Urgencias')

# Riesgo Cardiovascular segun el nivel lipídico y generacional (Scatter plot)
plt.subplot(1, 2, 2)
sns.scatterplot(data=datos_eda, x='Edad', y='Colesterol', hue='Estado_Cardiovascular', palette='Set1', alpha=0.6)
plt.title('Colesterol vs Edad de Pacientes')
plt.show()

---

## 3. Preprocesamiento Fisiológico (Preprocessing)

Las Redes Neuronales Diagnósticas odian:
1. Los "Pabéllon B" (Tienen que ser ceros o unos).
2. Omitir muestras en blando (Requieren que rellenemos inteligentemente los recuentos de glóbulos blancos).
3. Números hipertróficos. Si los Leucocitos marcan 15,000 en analíticas y la presión marca sólo 120, la IA puede enfermar y enfocarse solo en los Leucocitos.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Imputamos los exámenes pérdidos con la presión cardiáca 'Media' registrada en el hospital.
imputador = SimpleImputer(strategy='mean')
df_transformado[['Edad', 'Presion_Sistolica']] = imputador.fit_transform(df_transformado[['Edad', 'Presion_Sistolica']])

# One-Hot Encoding: Traducir la 'Enfermedad_Cardiaca' de texto a un 0 o un 1 que el algoritmo entienda biólogicamente.
encoder = OneHotEncoder(sparse_output=False, drop='first') # Usamos drop='first' para predecir si está enfermo sólo con bandera binaria=1
resultados_enfermos = encoder.fit_transform(df_transformado[['Enfermedad_Cardiaca']])
df_transformado['Enfermo_Num'] = resultados_enfermos

# Standard Scaler (Nivelamos variables extremas con la presión para que pesen igual dentro de la topografía neuronal).
escalador = StandardScaler()
df_transformado[['Edad', 'Glucosa_Sangre', 'Presion_Sistolica']] = escalador.fit_transform(df_transformado[['Edad', 'Glucosa_Sangre', 'Presion_Sistolica']])

print("\n--- TABLA LISTA (100% Optimizada en IA Medica) ---")
display(df_transformado.drop(columns=['Enfermedad_Cardiaca', 'Paciente_ID']))

---

## 4. Pipeline de Predicción del Diagnóstico (scikit-learn)

En lugar de limpiar todo este expediente manual que genera *Data Leakage*, construimos un `Pipeline` algorítmico como un Mega-Scanner a la cual entra el expediente original con errores y arroja directemente una probabilidad de cáncer de mama maligno o de infarto inminente.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

# Datos puramente extraidos, que contienen agujeros. (La Y es nuestra etiqueta a predecir)
X = df_extraido[['Edad', 'Glucosa_Sangre', 'PresionArterial']]
y = df_extraido['Enfermedad_Cardiaca']

X_limpio = X.copy()
# Hack rápido para limpiar la Presion a Númerico antes de meter al pipeline estrictamente tabular
X_limpio['PresionArterial'] = pd.to_numeric(X_limpio['PresionArterial'].str.replace('mmHg', ''), errors='coerce')

# Flujos o Sub-Pipelines de Diagnóstico
flujo_vitales_numericos = Pipeline(steps=[
    ('imputar_medias_hospital', SimpleImputer(strategy='mean')),
    ('escalar_zscore', StandardScaler())
])

# El robot preprocesador reúne los signos vitales
preprocesador_medico = ColumnTransformer(transformers=[
    ('vitales', flujo_vitales_numericos, ['Edad', 'Glucosa_Sangre', 'PresionArterial'])
])

# Finalmente pasamos el paciente del Scanner(Preprocesador) directo al Medico Especialista IA (RF Classifier)
hospital_IA_pipeline = Pipeline(steps=[
    ('preprocesamiento', preprocesador_medico),
    ('doctor_ia', RandomForestClassifier(random_state=42))
])

# Metemos a 5 pacientes de prueba (incluso con nulos)
hospital_IA_pipeline.fit(X_limpio, y)
print("\n\n¡El Doctor basado en IA está entrenado con este Pipeline quirurgicamente perfecto!")
print("Cualquier nuevo electrocardiograma o medición entrará automáticamente, se limpiará y dirá si requiere cirugía o no.")